In [4]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas tqdm neo4j

The stucture below suggest that instead of relying on OKCUPID profile vocabulary, llms would generate a travel-oriented semantic representation that can later match destination/activity descriptions.

In [5]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import torch
from tqdm.auto import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

CSV_PATH = Path(
    "/content/drive/My Drive/IRS Project (1)/Profiles/okcupid_profiles.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/My Drive/IRS Project (1)/Profiles_Out"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

JSONL_PATH = OUTPUT_DIR / "travel_profiles.jsonl"
CSV_OUTPUT_PATH = OUTPUT_DIR / "travel_profiles.csv"
ERROR_PATH = OUTPUT_DIR / "travel_profile_errors.jsonl"

# OpenAI model
MODEL_NAME = "gpt-5-mini"

# None = process all eligible profiles
MAX_PROFILES = None

MIN_TEXT_LENGTH = 100

print("Output:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output: /content/drive/My Drive/IRS Project (1)/Profiles_Out


In [6]:
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

essay_columns = [
    col for col in df.columns
    if col.startswith("essay")
]

print("Essay columns:", essay_columns)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/IRS Project (1)/Profiles/okcupid_profiles.csv'

In [ ]:
df = df.copy()

#Create an anonymous ID

df["profile_id"] = [
    f"P{i:06d}"
    for i in range(1, len(df) + 1)
]

In [ ]:
#Clean and combine essays as evidence, but remove noise

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Email addresses
    text = re.sub(
        r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        " ",
        text
    )

    # Excess whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def combine_essays(row):

    sections = []

    for col in essay_columns:

        text = clean_text(row[col])

        if text:
            sections.append(
                f"{col.upper()}:\n{text}"
            )

    return "\n\n".join(sections)


df["essay_text"] = df.apply(
    combine_essays,
    axis=1
)

df = df[
    df["essay_text"].str.len() >= MIN_TEXT_LENGTH
].copy()

print("Usable profiles:", len(df))

df[["profile_id", "essay_text"]].head()

Open-source LLM

In [ ]:
!pip uninstall -y openai
!rm -rf /usr/local/lib/python3.13/dist-packages/openai
!rm -rf /usr/local/lib/python3.13/dist-packages/openai-*.dist-info
!pip install --no-cache-dir -U openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("TOGETHER_API_KEY")
)

In [ ]:
def build_prompt(essay_text):

    return f"""
Create a structured TravelProfile from the profile essays below.

The output schema already defines the allowed concepts for:
- activities
- interests
- place preferences
- travel styles

You MUST select only from those predefined concepts.
Do not create new concept names, categories, labels, or synonyms.

GENERAL RULES:

1. Extract travel-relevant preferences only.

2. Prefer explicit information over inferred information.

3. Use "explicit" when the preference is directly supported by
   something the person wrote.

4. Use "inferred" only when the preference is a direct and
   conservative consequence of explicit evidence.

5. Do not infer a preference merely because it is commonly
   associated with another preference.

For example:

"I ski every winter"
→ skiing = explicit
→ mountains = inferred
→ active = inferred

"I love trying new restaurants"
→ restaurant_dining = explicit
→ food = inferred

But do NOT automatically infer:
→ culinary travel
→ urban exploration
→ nightlife
→ luxury travel

6. If there is insufficient evidence for a concept, omit it.
   Empty lists are better than unsupported assumptions.

7. Confidence must be between 0 and 1 and represents how certain
   you are that the essay supports the classification.

8. preference_strength represents how strongly the person's text
   indicates the preference:
   - low
   - moderate
   - high

Confidence and preference strength are different.

9. Evidence must contain short phrases grounded in the original
   essays. Do not invent evidence.

10. Avoid duplicate concepts. If multiple pieces of evidence support
    the same concept, return the concept once and combine the evidence.

11. mentioned_places should contain only actual geographic places
    explicitly mentioned in the essays. Do not infer destinations.

12. Avoidances should only represent activities, interests, or
    environments that the person explicitly dislikes or avoids.

13. Social preferences should only be included when supported by
    evidence. Do not infer solo, couple, or group preferences without
    evidence.

14. Do not infer sensitive characteristics such as gender, sexual
    orientation, ethnicity, religion, political beliefs, health,
    relationship status, or socioeconomic status.


TRAVEL INTEREST SUMMARY:

Create a short summary describing the person's strongest
travel-relevant interests.

The summary may synthesize multiple supported preferences but must
not introduce unsupported characteristics.


TRAVEL EMBEDDING TEXT:

Generate a rich hypothetical description of travel experiences,
activities, attractions, and environments that would likely match
the person's supported preferences.

This field is intended for semantic/vector retrieval.

It may expand concepts into closely related travel vocabulary,
but must remain grounded in the extracted preferences.

For example, if supported preferences include:

skiing + hiking + food + mountains

a suitable representation could be:

"Active outdoor travel involving skiing, winter sports, mountain
environments, hiking trails and nature experiences, combined with
local food, restaurant discovery and culinary experiences."

Do not:
- invent specific destinations
- invent budget preferences
- invent luxury preferences
- invent accommodation preferences
- invent transportation preferences
- introduce unrelated activities


PROFILE ESSAYS:

{essay_text}
"""

Travel Profile Schema

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal


# ============================================================
# CONTROLLED VOCABULARY
# ============================================================

ActivityType = Literal[
    "hiking",
    "walking",
    "cycling",
    "running",
    "skiing",
    "swimming",
    "water_sports",
    "camping",
    "wildlife_viewing",
    "sightseeing",
    "museum_visiting",
    "historical_sites",
    "shopping",
    "restaurant_dining",
    "food_tours",
    "cooking_experiences",
    "nightlife",
    "live_music",
    "festivals",
    "photography",
    "wellness",
    "adventure_sports"
]


InterestType = Literal[
    "nature",
    "food",
    "history",
    "art",
    "culture",
    "music",
    "architecture",
    "technology",
    "sports",
    "fashion",
    "photography",
    "wildlife",
    "literature",
    "wellness"
]


PlaceType = Literal[
    "mountains",
    "beaches",
    "forests",
    "countryside",
    "desert",
    "lakes",
    "islands",
    "urban",
    "small_towns",
    "historical_areas",
    "natural_areas"
]


TravelStyleType = Literal[
    "active",
    "adventure",
    "cultural",
    "culinary",
    "nature",
    "relaxation",
    "social",
    "nightlife",
    "urban_exploration",
    "slow_travel"
]


# ============================================================
# PREFERENCE TYPES
# ============================================================

class ActivityPreference(BaseModel):
    concept: ActivityType
    confidence: float = Field(ge=0, le=1)
    preference_strength: Literal[
        "low",
        "moderate",
        "high"
    ]
    evidence: List[str]
    source_type: Literal[
        "explicit",
        "inferred"
    ]


class InterestPreference(BaseModel):
    concept: InterestType
    confidence: float = Field(ge=0, le=1)
    preference_strength: Literal[
        "low",
        "moderate",
        "high"
    ]
    evidence: List[str]
    source_type: Literal[
        "explicit",
        "inferred"
    ]


class PlacePreference(BaseModel):
    concept: PlaceType
    confidence: float = Field(ge=0, le=1)
    preference_strength: Literal[
        "low",
        "moderate",
        "high"
    ]
    evidence: List[str]
    source_type: Literal[
        "explicit",
        "inferred"
    ]


class TravelStylePreference(BaseModel):
    concept: TravelStyleType
    confidence: float = Field(ge=0, le=1)
    preference_strength: Literal[
        "low",
        "moderate",
        "high"
    ]
    evidence: List[str]
    source_type: Literal[
        "explicit",
        "inferred"
    ]


# ============================================================
# FINAL TRAVEL PROFILE
# ============================================================

class TravelProfile(BaseModel):

    travel_interest_summary: str

    activities: List[ActivityPreference]

    interests: List[InterestPreference]

    place_preferences: List[PlacePreference]

    travel_styles: List[TravelStylePreference]

    mentioned_places: List[str]

    travel_embedding_text: str

In [ ]:
def generate_profile(essay_text):

    response = client.responses.parse(
        model=MODEL_NAME,
        input=build_prompt(essay_text),
        text_format=TravelProfile
    )

    profile = response.output_parsed

    if profile is None:
        raise ValueError(
            "No valid TravelProfile returned."
        )

    return profile.model_dump()

Person ──LIKES_ACTIVITY──► Activity {concept: "nightlife"}


Person ──INTERESTED_IN───► Interest {concept: "music"}


Person ──PREFERS_PLACE───► PlaceType {concept: "mountains"}

In [ ]:
# ============================================================
# TEST ONE PROFILE — BEFORE AND AFTER
# ============================================================

# Select one profile
sample = df.iloc[5]

# ------------------------------------------------------------
# BEFORE: Original profile essays
# ------------------------------------------------------------

print("=" * 100)
print("BEFORE — ORIGINAL PROFILE")
print("=" * 100)

print("Profile ID:", sample["profile_id"])
print()

print(sample["essay_text"])


# ------------------------------------------------------------
# GENERATE TRAVEL PROFILE
# ------------------------------------------------------------

profile = generate_profile(
    sample["essay_text"]
)

profile["profile_id"] = sample["profile_id"]


# ------------------------------------------------------------
# AFTER: Generated TravelProfile
# ------------------------------------------------------------

print("\n\n")
print("=" * 100)
print("AFTER — GENERATED TRAVEL PROFILE")
print("=" * 100)

print(
    json.dumps(
        profile,
        indent=2,
        ensure_ascii=False
    )
)

Processing & Checkpoints

In [ ]:
# ============================================================
# PROCESS ALL PROFILES
# ============================================================

import time

MAX_RETRIES = 5


# ------------------------------------------------------------
# LOAD EXISTING CHECKPOINT
# ------------------------------------------------------------

processed_ids = set()

if JSONL_PATH.exists():

    with open(
        JSONL_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:
                record = json.loads(line)
                processed_ids.add(
                    record["profile_id"]
                )

            except Exception:
                pass


print("Already processed:", len(processed_ids))


# ------------------------------------------------------------
# SELECT REMAINING PROFILES
# ------------------------------------------------------------

remaining_df = df[
    ~df["profile_id"].isin(processed_ids)
].copy()

# Optional safeguard
remaining_df = remaining_df[
    remaining_df["essay_text"].str.len()
    >= MIN_TEXT_LENGTH
].copy()


if MAX_PROFILES is not None:
    remaining_df = remaining_df.head(MAX_PROFILES)


print("Eligible profiles:", len(df))
print("Remaining to process:", len(remaining_df))

In [ ]:
def generate_with_retry(
    essay_text,
    max_retries=MAX_RETRIES
):

    for attempt in range(max_retries):

        try:

            return generate_profile(
                essay_text
            )

        except Exception as e:

            if attempt == max_retries - 1:
                raise

            wait_time = min(
                2 ** attempt,
                30
            )

            print(
                f"\nTemporary error: {e}"
                f"\nRetrying in {wait_time}s..."
            )

            time.sleep(wait_time)

In [ ]:
successful = 0
failed = 0


for _, row in tqdm(
    remaining_df.iterrows(),
    total=len(remaining_df),
    desc="Generating TravelProfiles"
):

    profile_id = row["profile_id"]

    try:

        profile = generate_with_retry(
            row["essay_text"]
        )

        profile["profile_id"] = profile_id

        # Save immediately
        with open(
            JSONL_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    profile,
                    ensure_ascii=False
                )
                + "\n"
            )

        processed_ids.add(profile_id)

        successful += 1


    except Exception as e:

        failed += 1

        error_record = {
            "profile_id": profile_id,
            "error": str(e)
        }

        with open(
            ERROR_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    error_record,
                    ensure_ascii=False
                )
                + "\n"
            )


print("\nProcessing complete.")
print("Successful:", successful)
print("Failed:", failed)
print("Total saved:", len(processed_ids))